<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 03-B — Logistic Regression for Text Classification

**Course:** Natural Language Processing  
**Session:** 3 / 8  
**Duration:** ~40 min

---

## Objectives

By the end of this session you will be able to:

- Train a Logistic Regression classifier on TF-IDF features with sklearn
- Understand and tune the regularisation parameter C
- Inspect the weight vectors to understand what the model has learned
- Compare LR against your best Naive Bayes from TP-A
- Bonus: implement gradient descent on a toy problem from scratch

---

## Roadmap

| Part | Task | Deliverable |
|------|------|-------------|
| 1 | Data loading (same split as TP-A) | Shared preprocessing |
| 2 | Baseline LR with sklearn | F1, confusion matrix |
| 3 | Regularisation sweep | Accuracy vs C plot |
| 4 | Feature analysis — learned weights | Top weights per class |
| 5 | NB vs LR comparison | Summary table |
| Bonus | Sigmoid + gradient descent from scratch | Mini implementation |

Each part ends with a **📝 Your analysis** cell.

In [ ]:
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
SEED = 42
np.random.seed(SEED)
AG_NEWS_LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
print("Imports OK")

---

## Part 1 — Data loading

**Important:** use the same preprocessing function and the same `SEED` as in TP-A.
The split must be identical so that comparisons between NB and LR are valid.

In [ ]:
def preprocess_text(text: str) -> str:
    """Minimal 5-step text normalisation pipeline.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_ag_news(
    val_size: float = 0.1,
    seed: int = SEED,
) -> tuple[list[str], list[str], list[str], list[int], list[int], list[int]]:
    """Load AG News, preprocess, and split into train/val/test.

    Parameters
    ----------
    val_size : float
        Fraction of training data used for validation.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple
        (train_texts, val_texts, test_texts, train_labels, val_labels, test_labels)
    """
    raw = load_dataset("ag_news")
    rng = np.random.default_rng(seed)

    train_texts_raw = [r["text"] for r in raw["train"]]
    train_labels_raw = [r["label"] for r in raw["train"]]
    test_texts = [preprocess_text(r["text"]) for r in raw["test"]]
    test_labels = [r["label"] for r in raw["test"]]

    n = len(train_texts_raw)
    idx = rng.permutation(n)
    split = int(n * (1 - val_size))
    train_idx, val_idx = idx[:split], idx[split:]

    train_texts  = [preprocess_text(train_texts_raw[i]) for i in train_idx]
    train_labels = [train_labels_raw[i]                 for i in train_idx]
    val_texts    = [preprocess_text(train_texts_raw[i]) for i in val_idx]
    val_labels   = [train_labels_raw[i]                 for i in val_idx]

    return train_texts, val_texts, test_texts, train_labels, val_labels, test_labels


train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = load_ag_news()
label_names = [AG_NEWS_LABELS[i + 1] for i in range(4)]
print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")

---

## Part 2 — Baseline Logistic Regression

Build a pipeline: `TfidfVectorizer` → `LogisticRegression`.

Key parameters to be aware of:
- `C`: inverse regularisation strength (larger C = less regularisation)
- `solver='lbfgs'`: efficient for multi-class problems with dense features
- `multi_class='multinomial'`: uses softmax instead of one-vs-rest
- `max_iter=1000`: LR may need more iterations than the default (100)

### 2.1 — Implement `build_lr_pipeline`

In [ ]:
def build_lr_pipeline(
    C: float = 1.0,
    penalty: str = "l2",
    vectorizer: str = "tfidf",
    max_features: int | None = 50_000,
    ngram_range: tuple[int, int] = (1, 1),
    max_iter: int = 1000,
) -> Pipeline:
    """Build a sklearn Pipeline: TF-IDF → LogisticRegression.

    Parameters
    ----------
    C : float
        Inverse regularisation strength. Larger values = less regularisation.
    penalty : {'l1', 'l2', 'none'}
        Regularisation type. Use solver='saga' for L1.
    vectorizer : {'tfidf', 'count'}
        Feature extraction method.
    max_features : int or None
        Maximum vocabulary size.
    ngram_range : tuple[int, int]
        N-gram range for the vectoriser.
    max_iter : int
        Maximum number of solver iterations.

    Returns
    -------
    Pipeline
        Unfitted sklearn Pipeline.

    Raises
    ------
    ValueError
        If vectorizer is not 'tfidf' or 'count'.
    """
    # YOUR CODE HERE
    # Note: for L1 penalty, use solver='saga'. For L2, solver='lbfgs' works fine.
    raise NotImplementedError


def evaluate_classifier(
    y_true: list[int],
    y_pred: np.ndarray,
    label_names: list[str],
    title: str = "Evaluation",
) -> dict[str, float]:
    """Print classification report and plot confusion matrix.

    Parameters
    ----------
    y_true : list[int]
        Ground-truth labels.
    y_pred : np.ndarray
        Predicted labels.
    label_names : list[str]
        Human-readable class names.
    title : str
        Title for the confusion matrix figure.

    Returns
    -------
    dict[str, float]
        Dictionary with 'accuracy' and 'macro_f1'.
    """
    report = classification_report(
        y_true, y_pred, target_names=label_names, output_dict=True
    )
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(classification_report(y_true, y_pred, target_names=label_names))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return {"accuracy": report["accuracy"], "macro_f1": report["macro avg"]["f1-score"]}

In [ ]:
# ── Train and evaluate baseline LR ────────────────────────────────────────────
pipe_lr = build_lr_pipeline(C=1.0)
pipe_lr.fit(train_texts, train_labels)
val_pred_lr = pipe_lr.predict(val_texts)

results_lr = evaluate_classifier(
    val_labels, val_pred_lr, label_names,
    title="LR (TF-IDF, C=1.0, L2) — validation",
)

📝 **Your analysis (2.1)**

1. What macro F1 did LR achieve? Compare to your best NB from TP-A.
2. Which class is hardest? Is it the same as for NB?
3. Look at the confusion matrix. Which classes are most confused?

*Your answer here.*

---

## Part 3 — Regularisation sweep

The parameter C controls regularisation:
- Small C → strong regularisation → simpler model (high bias, low variance)
- Large C → weak regularisation → complex model (low bias, high variance)

### 3.1 — L2 sweep

In [ ]:
def regularisation_sweep(
    C_values: list[float],
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
    penalty: str = "l2",
) -> pd.DataFrame:
    """Evaluate LogisticRegression across a range of regularisation strengths.

    Parameters
    ----------
    C_values : list[float]
        Values of C to evaluate.
    train_texts : list[str]
        Training documents.
    train_labels : list[int]
        Training labels.
    val_texts : list[str]
        Validation documents.
    val_labels : list[int]
        Validation labels.
    penalty : str
        Regularisation type ('l1' or 'l2').

    Returns
    -------
    pd.DataFrame
        Columns: ['C', 'val_accuracy', 'val_macro_f1', 'train_macro_f1'].
        Include train_macro_f1 to detect overfitting.
    """
    # YOUR CODE HERE
    raise NotImplementedError


C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]
sweep_df = regularisation_sweep(
    C_values, train_texts, train_labels, val_texts, val_labels, penalty="l2"
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sweep_df["C"], sweep_df["val_macro_f1"],   marker="o", label="val macro F1")
ax.semilogx(sweep_df["C"], sweep_df["train_macro_f1"], marker="s", linestyle="--", label="train macro F1")
ax.set_xlabel("C (inverse regularisation)")
ax.set_ylabel("macro F1")
ax.set_title("Logistic Regression — L2 regularisation sweep")
ax.legend()
plt.tight_layout()
plt.show()

best_C = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "C"]
print(f"Best C = {best_C}")
print(sweep_df.to_string(index=False))

### 3.2 — L1 vs L2 comparison

L1 regularisation produces sparse models (many weights = 0). This is interesting for NLP:
it effectively selects a subset of features.

In [ ]:
def compare_penalties(
    C: float,
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
) -> pd.DataFrame:
    """Compare L1 and L2 regularisation for Logistic Regression.

    Also reports model sparsity (fraction of zero weights).

    Parameters
    ----------
    C : float
        Regularisation strength (same for both penalties).
    train_texts, train_labels, val_texts, val_labels : lists
        Data splits.

    Returns
    -------
    pd.DataFrame
        Columns: ['penalty', 'val_macro_f1', 'n_zero_weights', 'sparsity']
    """
    # YOUR CODE HERE
    # Hint: after fitting, pipe[1].coef_ has shape (n_classes, vocab_size)
    # sparsity = fraction of elements in coef_ that are exactly 0
    raise NotImplementedError


penalty_df = compare_penalties(
    best_C, train_texts, train_labels, val_texts, val_labels
)
print(penalty_df.to_string(index=False))

📝 **Your analysis (3)**

1. What is the optimal C for L2? What happens at very small C and very large C?
2. Where do the train and val curves diverge? What does this tell you about overfitting?
3. How sparse is the L1 model? What fraction of features are actually used?
4. Does L1 or L2 produce better validation F1? Does this surprise you?

*Your answer here.*

---

## Part 4 — Feature analysis: learned weights

Unlike Naive Bayes (which stores log-probabilities), Logistic Regression stores
a weight matrix of shape `(n_classes, vocab_size)`. A large positive weight for
class c means that word strongly predicts class c.

In [ ]:
def top_weights_per_class(
    pipeline: Pipeline,
    label_names: list[str],
    n: int = 20,
) -> dict[str, pd.DataFrame]:
    """Extract the n words with the largest positive and negative weights per class.

    Parameters
    ----------
    pipeline : Pipeline
        Fitted sklearn Pipeline (vectoriser + LogisticRegression).
    label_names : list[str]
        Human-readable class names in label order.
    n : int
        Number of top/bottom features per class.

    Returns
    -------
    dict[str, pd.DataFrame]
        class_name → DataFrame with columns ['word', 'weight', 'direction'].
        'direction' is '+' for positive weights, '-' for negative.
    """
    # YOUR CODE HERE
    # Hint: pipeline[0].get_feature_names_out() → vocabulary
    # pipeline[1].coef_ has shape (n_classes, vocab_size)
    raise NotImplementedError


def plot_top_weights(
    weights: dict[str, pd.DataFrame],
    n_plot: int = 15,
) -> None:
    """Horizontal bar plot of top positive and negative weights per class.

    Parameters
    ----------
    weights : dict[str, pd.DataFrame]
        Output of `top_weights_per_class`.
    n_plot : int
        Number of words (positive + negative) to show per class.
    """
    # YOUR CODE HERE
    # Suggestion: use a diverging colour scheme (positive=blue, negative=red)
    raise NotImplementedError


pipe_best = build_lr_pipeline(C=best_C)
pipe_best.fit(train_texts, train_labels)
weights = top_weights_per_class(pipe_best, label_names, n=20)

for name, df in weights.items():
    print(f"\n{name} — top 10 positive:")
    print(df[df["direction"] == "+"].head(10).to_string(index=False))

plot_top_weights(weights)

### 4.2 — Compare NB informativeness scores vs LR weights

Do NB and LR agree on which words are most discriminative?

In [ ]:
def compare_nb_lr_features(
    nb_features: dict[str, pd.DataFrame],
    lr_weights: dict[str, pd.DataFrame],
    class_name: str,
    top_k: int = 20,
) -> pd.DataFrame:
    """Compute overlap between top-k NB and LR features for one class.

    Parameters
    ----------
    nb_features : dict[str, pd.DataFrame]
        Output of `most_informative_features` from TP-A (or re-implement it here).
    lr_weights : dict[str, pd.DataFrame]
        Output of `top_weights_per_class`.
    class_name : str
        The class to compare (must be in both dicts).
    top_k : int
        Number of top features to compare.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['word', 'in_nb', 'in_lr', 'nb_score', 'lr_weight'].
    """
    # YOUR CODE HERE
    raise NotImplementedError


# ── Re-implement or import NB features ───────────────────────────────────────
# If you have nb_features from TP-A in memory, use them directly.
# Otherwise, re-fit a NB pipeline here (same alpha as before).
# YOUR CODE HERE

# comparison = compare_nb_lr_features(nb_features, weights, class_name="Sports")
# print(comparison.to_string(index=False))

📝 **Your analysis (4)**

1. What do the top positive weights look like? Do they make intuitive sense?
2. What do the top negative weights look like for a given class? For Sports, what are the words that most push the model away from predicting Sports?
3. Do NB and LR agree on the most discriminative words? Where do they disagree, and why might that be?

*Your answer here.*

---

## Part 5 — NB vs LR comparison + n-gram ablation

Bring together the results from both TPs into one comparison table.

In [ ]:
def full_ablation(
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
    best_C: float,
    best_nb_alpha: float = 0.1,
) -> pd.DataFrame:
    """Run all model/feature combinations and collect validation metrics.

    Parameters
    ----------
    train_texts, train_labels, val_texts, val_labels : lists
        Data splits.
    best_C : float
        Best regularisation parameter for LR (from Part 3).
    best_nb_alpha : float
        Best smoothing parameter for NB (from TP-A).

    Returns
    -------
    pd.DataFrame
        Columns: ['model', 'features', 'val_macro_f1', 'val_accuracy']
    """
    from sklearn.naive_bayes import MultinomialNB

    configs = [
        # (label, pipeline)
        # YOUR CODE HERE — include at least:
        # NB count unigrams, NB tfidf unigrams, NB tfidf bigrams
        # LR tfidf unigrams, LR tfidf bigrams
    ]

    rows = []
    for label, pipe in configs:
        pipe.fit(train_texts, train_labels)
        preds = pipe.predict(val_texts)
        report = classification_report(val_labels, preds, output_dict=True)
        rows.append({
            "model": label.split("|")[0].strip(),
            "features": label.split("|")[1].strip() if "|" in label else "",
            "val_macro_f1": round(report["macro avg"]["f1-score"], 4),
            "val_accuracy": round(report["accuracy"], 4),
        })
    return pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)


# abl_df = full_ablation(train_texts, train_labels, val_texts, val_labels, best_C=best_C)
# print(abl_df.to_string(index=False))

📝 **Your analysis (5)**

1. Which configuration is best overall? By how much does LR beat NB?
2. Do bigrams help LR as much as they helped NB?
3. Where is the best cost-efficiency tradeoff (performance vs model complexity)?

*Your answer here.*

---

## Part 6 — Final test evaluation

Evaluate your best LR configuration on the test set (once, no further tuning).

In [ ]:
# YOUR CODE HERE
# Train best LR on train_texts, predict on test_texts, call evaluate_classifier
# Record the result in the summary table below.

---

## Bonus — Gradient descent from scratch

Implement gradient descent for binary logistic regression on a 2D synthetic dataset.
This is a simplified version of what sklearn does internally.

The goal is to gain intuition for:
- How the sigmoid maps scores to probabilities
- How the weight update moves the decision boundary
- How learning rate η affects convergence

In [ ]:
# ── Synthetic 2D dataset ──────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)
n_pos, n_neg = 200, 200
X_pos = rng.multivariate_normal([2, 2], [[1, 0.5], [0.5, 1]], n_pos)
X_neg = rng.multivariate_normal([-2, -2], [[1, -0.3], [-0.3, 1]], n_neg)

X = np.vstack([X_pos, X_neg]).astype(np.float64)
y = np.array([1] * n_pos + [0] * n_neg)

shuffle_idx = rng.permutation(len(y))
X, y = X[shuffle_idx], y[shuffle_idx]

# Add bias column
X_bias = np.hstack([np.ones((len(X), 1)), X])
print(f"X shape: {X_bias.shape}, y shape: {y.shape}")

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """Compute the sigmoid function element-wise.

    Parameters
    ----------
    z : np.ndarray
        Input array of any shape.

    Returns
    -------
    np.ndarray
        Element-wise σ(z) = 1 / (1 + exp(−z)).

    Examples
    --------
    >>> sigmoid(np.array([0.0]))
    array([0.5])
    >>> sigmoid(np.array([-100.0, 0.0, 100.0]))
    array([~0., 0.5, ~1.])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cross_entropy_loss(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    """Compute binary cross-entropy loss.

    Parameters
    ----------
    y_true : np.ndarray of shape (N,)
        Ground-truth binary labels (0 or 1).
    y_pred : np.ndarray of shape (N,)
        Predicted probabilities in (0, 1).

    Returns
    -------
    float
        Mean binary cross-entropy loss.

    Notes
    -----
    Clip y_pred to [1e-15, 1 - 1e-15] to avoid log(0).

    Examples
    --------
    >>> cross_entropy_loss(np.array([1, 0]), np.array([0.9, 0.1]))
    ~0.105
    """
    # YOUR CODE HERE
    raise NotImplementedError


def gradient_descent_lr(
    X: np.ndarray,
    y: np.ndarray,
    learning_rate: float = 0.1,
    n_epochs: int = 200,
) -> tuple[np.ndarray, list[float]]:
    """Train binary logistic regression via gradient descent.

    Parameters
    ----------
    X : np.ndarray of shape (N, D+1)
        Feature matrix with bias column prepended.
    y : np.ndarray of shape (N,)
        Binary labels (0 or 1).
    learning_rate : float
        Step size η for each gradient update.
    n_epochs : int
        Number of passes over the full dataset.

    Returns
    -------
    w : np.ndarray of shape (D+1,)
        Learned weight vector (including bias as w[0]).
    losses : list[float]
        Cross-entropy loss at the end of each epoch.

    Notes
    -----
    Update rule: w ← w − η/N · Xᵀ (ŷ − y)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# ── Train ─────────────────────────────────────────────────────────────────────
w_learned, losses = gradient_descent_lr(X_bias, y, learning_rate=0.1, n_epochs=200)

# ── Plot loss curve ───────────────────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title("Gradient descent — loss curve")
plt.tight_layout()
plt.show()

# ── Accuracy ──────────────────────────────────────────────────────────────────
y_pred_binary = (sigmoid(X_bias @ w_learned) >= 0.5).astype(int)
acc = (y_pred_binary == y).mean()
print(f"Training accuracy: {acc:.4f}")

In [ ]:
def plot_decision_boundary(
    X: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    title: str = "Logistic Regression decision boundary",
) -> None:
    """Plot 2D scatter with the learned linear decision boundary.

    Parameters
    ----------
    X : np.ndarray of shape (N, 2)
        Original 2D features (without bias column).
    y : np.ndarray of shape (N,)
        Binary labels.
    w : np.ndarray of shape (3,)
        Weight vector [bias, w1, w2].
    title : str
        Figure title.

    Notes
    -----
    Decision boundary: w[0] + w[1]*x1 + w[2]*x2 = 0
    → x2 = -(w[0] + w[1]*x1) / w[2]
    """
    # YOUR CODE HERE
    raise NotImplementedError


plot_decision_boundary(X, y, w_learned)

📝 **Bonus analysis**

1. At what epoch does the loss stop decreasing significantly?
2. Try learning_rate = 1.0 and learning_rate = 0.001. What happens in each case?
3. The update rule `w ← w − η/N · Xᵀ(ŷ − y)` is a full-batch gradient. How would you modify it for stochastic gradient descent (one example at a time)?

*Your answer here.*

---

## Summary — Running comparison table

Fill in after completing all parts.

In [ ]:
results_table = pd.DataFrame([
    {"session": "01", "model": "Majority baseline",       "features": "none",           "val_macro_f1": 0.25,  "test_macro_f1": 0.25},
    # Add your NB results from TP-A
    # Add your LR results from TP-B
])
print(results_table.sort_values("val_macro_f1", ascending=False).to_string(index=False))